# Temperature Anomalies per Station

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/02_temperature_anomalies.en.ipynb)

**Learning objective**: In this notebook you will compute each weather station's temperature anomaly relative to the network's spatial mean, both as a tabular/chart summary (`lcz_anomaly`) and as a continuous kriged surface (`lcz_anomaly_map`), using Berlin as the reference city.

## Why anomalies? Isolating the "urban effect" from synoptic weather

Raw air temperature at any single station is dominated by synoptic-scale weather patterns — cold fronts, air masses, the seasonal cycle — that affect every station across a region almost equally. If you compare two stations directly by absolute temperature, the local climate signal we actually care about in urban climatology is buried under this regional "noise" that is common to all of them.

**Anomaly** solves this: at every timestamp, we subtract the spatial mean across all network stations from the value observed at each individual station. What remains is exactly the component that is *not* explained by regional weather — i.e., how much warmer or cooler that station consistently is than its peers, controlling for the day's meteorological variability. This "spatial residual" is the signature of the **local urban effect**: impervious surface cover, lack of vegetation, reduced sky view factor (urban canyons), roughness, and the thermal capacity of built materials.

This reasoning connects directly to the Local Climate Zone (LCZ) scheme of Stewart & Oke (2012): stations in compact, impervious LCZs (LCZ 1-3, 8-10) tend to show persistent positive anomalies, while stations in vegetated LCZs or near water bodies (LCZ 11-17) tend to show negative anomalies. The LCZ4r package (Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) formalizes this workflow, joining a station's LCZ classification with its anomaly time series.

This notebook covers two complementary tools:

- **`lcz_anomaly`** — computes each station's mean anomaly (aggregated over time) and produces ranking charts (diverging bars, dots, lollipops, scatter). Ideal for answering "which stations are consistently warmer?" and for communicating results directly.
- **`lcz_anomaly_map`** — for each time step (or temporal group), krige the point anomalies onto a regular grid, producing a continuous raster. Uses the LCZ class as an external drift term in kriging when `LCZinterp=True`, which helps the interpolator respect the boundaries between urban and rural zones even in areas with few stations. Ideal for mapping the full spatial extent of the effect, not just at station points.

## Installation

Install LCZ4py with all optional extras (kriging, DuckDB, GeoArrow, etc.).

In [ ]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"

## Setup (repeated from notebook 01)

Since every notebook in this series must run independently in a fresh Colab session, we repeat the setup here: download Berlin's LCZ map and load weather station data (`lcz_data_berlin.csv`, with columns `date`, `station`, `airT`, `Latitude`, `Longitude`).

In [2]:
from LCZ4py import lcz_get_map
import pandas as pd

lcz_map_path = lcz_get_map(city="Berlin")
print(lcz_map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df.head()

06:24:24 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01 00:00:00,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01 00:00:00,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01 00:00:00,albrecht,8.04725,52.444594,13.348607
3,806407,2019-01-01 00:00:00,bamberger,8.27166,52.496494,13.337552
4,806408,2019-01-01 00:00:00,baruth,8.20000,52.061300,13.499700


## `lcz_anomaly` — ranking stations by anomaly

`lcz_anomaly(x, data_frame, var, station_id, plot_type=..., by=None, iplot=True, ...)` assigns each station to an LCZ class (from `x`, the LCZ raster), computes the time-mean of `var` at each station, and subtracts the overall spatial mean across the whole network — that difference is the anomaly. Key parameters:

- `var="airT"`, `station_id="station"` — column names for temperature and station identifier.
- `plot_type` — chart style: `"diverging_bar"` (default, horizontal bars diverging from zero), `"bar"` (vertical bars colored red/blue), `"dot"` (Cleveland dot plot connecting a station's mean to the reference), `"lollipop"` (vertical stems from zero to the anomaly), or `"scatter"` (LCZ class vs. mean temperature, dot size = |anomaly|).
- `by` — optionally facet by a temporal group (`"year"`, `"season"`, etc.).
- `iplot=False` — return only the `pl.DataFrame` of anomalies, no plot.
- `year=`/`month=`/... — date filters, same convention as `lcz_ts`.

The return value (when `iplot=True`) is an interactive Plotly figure; each station is labeled with its LCZ class and colored by the official LCZ palette. Stations at the positive extreme are local "heat islands"; at the negative extreme, "cool islands".

Below we demonstrate three chart styles: `diverging_bar`, `dot`, and `scatter`.

In [3]:
from LCZ4py import lcz_anomaly

fig_div = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="diverging_bar",
    year=2019,
)
fig_div.show()

**Interpretation**: bars extending to the right (positive values) are stations warmer than the network mean in 2019 — usually stations in compact/impervious LCZs. Bars extending left are cooler — typically in vegetated LCZs or near water bodies. Each bar's color reflects the station's LCZ class.

In [4]:
fig_dot = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="dot",
    year=2019,
)
fig_dot.show()

**Interpretation**: the Cleveland dot plot shows, for each station, its mean temperature (dot colored by LCZ) connected by a gray line to the reference value (the overall spatial mean, gray dot). The farther the colored dot is from the reference dot, the larger that station's anomaly — this view is useful for comparing absolute temperature magnitudes, not just the deviation.

In [5]:
fig_scatter = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="scatter",
    year=2019,
)
fig_scatter.show()

**Interpretation**: here the X axis is LCZ class and the Y axis is station mean temperature; marker size scales with |anomaly|. This quickly reveals whether the anomaly tracks LCZ class consistently (as expected from Stewart & Oke theory) or whether there are "outlier" stations within their own class — possibly due to poor sensor exposure, local microtopography, or proximity to anthropogenic heat sources not captured by the LCZ classification.

## `lcz_anomaly_map` — continuous spatial anomaly surface

While `lcz_anomaly` answers "which station is warmest", `lcz_anomaly_map` answers "what is the full spatial pattern of the anomaly across the whole city" — producing a continuous raster via parallel kriging instead of discrete station-point values.

For each time step (or group, if `by` is set), the function:
1. computes each station's point anomaly (value − network spatial mean at that instant);
2. interpolates those point anomalies onto a regular grid via kriging (`method="krige"`, using the `vg_model` variogram model), with LCZ class as an external drift covariate when `LCZinterp=True` and at least 2 distinct LCZ classes are present among the stations;
3. saves a multi-band GeoTIFF (one band per time step).

Because parallel kriging over many stations/time steps is heavy, we use a coarse spatial resolution here (`sp_res=500` m) and a short time window (`month=1, year=2019`, ~744 hourly steps) to keep this tutorial fast. In a real study, a finer `sp_res` (50-100 m) and a longer period would give more detail, at the cost of runtime.

In [6]:
from LCZ4py import lcz_anomaly_map

anomaly_map_path = lcz_anomaly_map(
    lcz_map_path, df,
    var="airT", station_id="station",
    sp_res=500.0,
    tp_res="hour",
    method="krige",
    LCZinterp=True,
    year=2019, month=1,
    n_jobs=-1,
)
print(anomaly_map_path)

/var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpsvjqjjgn.tif


### Visualizing the anomaly raster

The resulting GeoTIFF has one band per time step. We use `lcz_plot_map(..., data_type="continuous")` to display one of them as a continuous heatmap (rather than the categorical mode used for LCZ class maps).

In [7]:
from LCZ4py import lcz_plot_map

result = lcz_plot_map(
    anomaly_map_path,
    data_type="continuous",
    band=1,
    colorscale="RdBu_r",
    title="Spatial air temperature anomaly (Berlin, band 1)",
)
result.fig.show()

**Interpretation**: warm tones (red) mark areas where the interpolation predicts temperature above the network spatial mean at that instant — usually over the compact urban core; cool tones (blue) appear over parks, forests, and water bodies. Because the kriging uses LCZ class as external drift, boundaries between zones tend to appear sharper than in a purely geographic interpolation (IDW/RBF), reflecting underlying urban structure rather than just distance to stations.

**Tabular vs. spatial — when to use which**: `lcz_anomaly` is the right tool when you want an **objective station ranking** (e.g. for reports, or to flag which sensors deserve an exposure inspection). `lcz_anomaly_map` is the right tool when you need a **continuous surface** — e.g. to estimate the anomaly at a location without a station, to overlay with land use, or to produce a public-communication map of the heat island effect.

## Conclusion

In this notebook you learned to isolate the local urban effect from regional weather via the anomaly technique: subtracting the network's spatial mean at each instant leaves the persistent, location-specific signal — the signature of urban form (LCZ) on temperature. We saw the tabular/chart version (`lcz_anomaly`, useful for ranking stations) and the spatially-interpolated version (`lcz_anomaly_map`, useful for mapping the whole city via LCZ-drift kriging).

**Previous notebook**: `01_lcz_time_series` — LCZ-stratified time series.
**Next notebook**: `03_urban_heat_island` — direct quantification of Urban Heat Island (UHI) intensity comparing urban and rural stations.